# Part 1 — Leakage audit

Recompute what each shipment should have been charged under its contract, compare it with what was billed, and explain the gaps.

- Rules: `src/contracts.py` (terms + clause refs), `src/billing.py` (one function per rule)
- Audit: `src/audit.py` (expected vs billed per awb + reason); summaries: `src/report.py`
- This notebook only calls those functions and answers the questions. INR, 2 dp, excl. GST.
- **Underbilled**: expected > billed by more than ₹0.005. **Overbilled**: expected < billed by more than ₹0.005.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.audit import build_audit, by_merchant, by_reason, write_outputs
from src.report import (inr_indian, inr_western, signed, top3, totals, merchant_rankings,
                        root_causes, root_cause_lines, multi_cause_rows, not_leakage,
                        not_leakage_lines, sanity_checks)
from src.verify import run_unit_tests, verification_checks, edge_cases

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

df = pd.read_csv(ROOT / 'data' / 'shipments.csv', parse_dates=['ship_date'])
audit = build_audit(df)
audit['status'].value_counts()

status
ok             4953
underbilled    1493
overbilled      554
Name: count, dtype: int64

## Sanity checks
Run first: if any fails, the answers below are not trustworthy.

In [2]:
checks = sanity_checks(audit)
assert checks['passed'].all(), checks[~checks['passed']]
checks

,check,passed
0,sum(expected) - sum(billed) == underbilled - overbilled,True
1,per-merchant net sums to overall net,True
2,billed_total == billed_freight + billed_cod_fee (all rows),True
3,every mismatch has a known cause (0 'unexplained'),True
4,₹ attributed to causes sums to total net underbilling,True
5,status 'ok' <=> reason 'ok',True
6,sole-cause rows + multi-cause rows == all mismatched rows,True


## Q1–Q4. Underbilled / overbilled totals

In [3]:
t = totals(audit)
pd.DataFrame({
    'shipments': [t['underbilled_n'], t['overbilled_n']],
    'amount (Indian)': [inr_indian(t['underbilled_inr']), inr_indian(t['overbilled_inr'])],
    'amount (western)': [inr_western(t['underbilled_inr']), inr_western(t['overbilled_inr'])],
}, index=['underbilled', 'overbilled'])

,shipments,amount (Indian),amount (western)
underbilled,1493,"₹35,986.32","₹35,986.32"
overbilled,554,"₹6,290.93","₹6,290.93"


## Q5. Top 3 merchants by net underbilling
Net = underbilled ₹ − overbilled ₹ per merchant. Gross ranking shown too in case it differs.

In [4]:
net, gross = merchant_rankings(audit)
print('Net   (Indian): ', top3(net, 'net_underbilled_inr', inr_indian))
print('Net   (western):', top3(net, 'net_underbilled_inr', inr_western))
print('Gross (Indian): ', top3(gross, 'underbilled_inr', inr_indian))
print('Same order:', list(net.index[:3]) == list(gross.index[:3]))
net[['name', 'underbilled_n', 'underbilled_inr', 'overbilled_n', 'overbilled_inr', 'net_underbilled_inr']]

Net   (Indian):  M05 ₹12,671.20; M11 ₹6,207.30; M07 ₹5,878.80
Net   (western): M05 ₹12,671.20; M11 ₹6,207.30; M07 ₹5,878.80
Gross (Indian):  M05 ₹12,777.42; M07 ₹8,216.10; M11 ₹6,207.30
Same order: False


,name,underbilled_n,underbilled_inr,overbilled_n,overbilled_inr,net_underbilled_inr
merchant_id,,,,,,
M05,Pustak Ghar,540,12777.42,15,106.22,12671.20
M11,Shilp Decor,168,6207.30,0,0.00,6207.30
M07,Patang Apparel,411,8216.10,211,2337.30,5878.80
M08,Vaidya Wellness,238,5137.05,0,0.00,5137.05
M10,Rasoi Ware,19,507.78,0,0.00,507.78
M12,Nadi Naturals,14,491.40,0,0.00,491.40
M04,Moti Jewels,14,443.52,0,0.00,443.52
M06,Sitar Sounds,13,405.00,33,655.00,-250.00
M02,Tara Spices,25,580.45,55,859.75,-279.30


## Q5 (cont.). Root causes

Each shipment's gap is split across its causes by fixing one error at a time with the billing engine (`src/audit.py: impact_steps`): discount → zone → weight → COD fee. So the cause ₹ add up exactly to the total net. Positive = underbilled, negative = overbilled.

In [5]:
causes = root_causes(audit)
assert abs(causes['net_inr'].sum() - (t['underbilled_inr'] - t['overbilled_inr'])) < 0.01
causes.drop(columns='cause')

,description,merchants,first_date,last_date,months,shipments,with_other_causes,underbilled_inr,overbilled_inr,net_inr,avg_inr_per_shipment,example_awb
cluster,,,,,,,,,,,,
C1,"Zone: Hyderabad<->metro routes billed Zone D instead of C (Hyderabad is a metro, A.2 rule 4)","M01 56, M02 56, M03 48, M05 62, M06 36, M07 394",2026-07-01,2026-09-30,"Jul,Aug,Sep",652,240,0.00,7505.72,-7505.72,-11.51,SFX2708195638
C2,Weight: chargeable weight rounded DOWN to the 0.5 kg slab instead of up (A.3),M05 452,2026-07-01,2026-09-30,"Jul,Aug,Sep",452,58,10606.96,-0.00,10606.96,23.47,SFX2707812567
C3,Weight: M07 billed with the 0.5 kg minimum instead of its 1.0 kg minimum (M07-§2.3),M07 402,2026-07-01,2026-09-30,"Jul,Aug,Sep",402,181,9187.20,-0.00,9187.20,22.85,SFX2706513280
C4,Weight: volumetric weight computed with divisor 6000 (no contract uses 6000; A.3 = 5000),"M01 20, M02 25, M03 20, M04 14, M05 103, M06 13, M07 13, M08 23, M09 11, M10 19, M11 15, M12 14",2026-07-01,2026-09-30,"Jul,Aug,Sep",290,48,7963.04,-0.00,7963.04,27.46,SFX2702647376
C5,"COD: M08 always billed the flat INR 35 minimum, never 2.5% of COD value (M08-§3.1)",M08 223,2026-07-02,2026-09-30,"Jul,Aug,Sep",223,8,4458.75,-0.00,4458.75,19.99,SFX2706749514
C6,Weight: M11 volumetric weight used divisor 5000 instead of its 4000 (M11-§2.2),M11 153,2026-07-01,2026-09-30,"Jul,Aug,Sep",153,0,5422.60,-0.00,5422.60,35.44,SFX2705708918
C7,"Discount: M09 Sep shipments billed at the old 8% discount, not 12% (M09-Ann1.1)",M09 139,2026-09-01,2026-09-30,Sep,139,0,0.00,719.44,-719.44,-5.18,SFX2700856401
C8,"Weight: volumetric weight ignored, dead weight billed alone (A.3)",M05 5,2026-07-10,2026-07-23,Jul,5,0,282.00,-0.00,282.00,56.40,SFX2706056575


In [6]:
for line in root_cause_lines(causes):
    print(line, end='\n\n')
multi = multi_cause_rows(audit)
print(f'{len(multi)} shipments have 2+ causes; their ₹ is split between those causes above, not double-counted.')
print(f"Sum of cause impacts = {inr_indian(causes['net_inr'].sum())} = total net underbilling.")

C1. Zone: Hyderabad<->metro routes billed Zone D instead of C (Hyderabad is a metro, A.2 rule 4). Affects M01 56, M02 56, M03 48, M05 62, M06 36, M07 394; ship dates 01 Jul - 30 Sep 2026. 652 shipments (240 also have another cause). Impact: overbilled ₹7,505.72 (avg ₹11.51/shipment). e.g. SFX2708195638

C2. Weight: chargeable weight rounded DOWN to the 0.5 kg slab instead of up (A.3). Affects M05 452; ship dates 01 Jul - 30 Sep 2026. 452 shipments (58 also have another cause). Impact: underbilled ₹10,606.96 (avg ₹23.47/shipment). e.g. SFX2707812567

C3. Weight: M07 billed with the 0.5 kg minimum instead of its 1.0 kg minimum (M07-§2.3). Affects M07 402; ship dates 01 Jul - 30 Sep 2026. 402 shipments (181 also have another cause). Impact: underbilled ₹9,187.20 (avg ₹22.85/shipment). e.g. SFX2706513280

C4. Weight: volumetric weight computed with divisor 6000 (no contract uses 6000; A.3 = 5000). Affects M01 20, M02 25, M03 20, M04 14, M05 103, M06 13, M07 13, M08 23, M09 11, M10 19, M11 

### Looked like leakage but isn't
Counts only correctly billed shipments where a naive rule would give a different charge; `correct_minus_naive_inr` is how far off the naive rule would be.

In [7]:
leak_free = not_leakage(audit)
leak_free

,pattern,shipments,correct_minus_naive_inr,naive_rule,why_not_leakage
0,M02 COD fee = ₹0 in August,150,-4394.50,the normal M02 COD fee (M02-§3.1),Waived by M02-Ann1.1 for COD shipments booked 1-31 Aug 2026; fee resumes 1 Sep (Ann1.2).
1,Billed weight above declared weight,689,20201.38,pricing on declared weight,Re-measured values are final for billing (§1.2 in every agreement).
2,Volumetric weight above dead weight,1067,58398.08,pricing on dead weight only,Chargeable weight = higher of dead and volumetric (A.3).
3,"Mumbai->Pune billed Zone B, not C (Mumbai is a metro)",47,-976.12,Zone C pricing,Same state is checked before metro (A.2 rule 3 before rule 4); Pune is not a metro anyway.


In [8]:
for line in not_leakage_lines(leak_free):
    print('-', line)

- M02 COD fee = ₹0 in August: 150 correctly billed shipments (correct charge is ₹4,394.50 less than the normal M02 COD fee (M02-§3.1)). Waived by M02-Ann1.1 for COD shipments booked 1-31 Aug 2026; fee resumes 1 Sep (Ann1.2).
- Billed weight above declared weight: 689 correctly billed shipments (correct charge is ₹20,201.38 more than pricing on declared weight). Re-measured values are final for billing (§1.2 in every agreement).
- Volumetric weight above dead weight: 1,067 correctly billed shipments (correct charge is ₹58,398.08 more than pricing on dead weight only). Chargeable weight = higher of dead and volumetric (A.3).
- Mumbai->Pune billed Zone B, not C (Mumbai is a metro): 47 correctly billed shipments (correct charge is ₹976.12 less than Zone C pricing). Same state is checked before metro (A.2 rule 3 before rule 4); Pune is not a metro anyway.


## Q6. How I checked my numbers

Three independent lines of evidence (all computed here, `src/verify.py`):
1. **Unit tests** — each billing rule pinned with hand-computed values.
2. **Second implementation** — a column-wise pandas/numpy recomputation that reads the same contract tables but does not call `src/billing.py`; it must agree on every row.
3. **Reconciliation** — where billing was right, the engine reproduces it to the paisa; where it was wrong, a known billing error reproduces the billed value exactly.

In [9]:
tests = run_unit_tests()
assert tests['failed'] == 0, tests['summary']
tests['summary']

'56 passed in 0.49s'

In [10]:
verification = verification_checks(audit)
assert verification['passed'].all(), verification[~verification['passed']]
verification

,check,evidence,passed
0,Independent vectorised recomputation agrees with engine,"zone 7,000/7,000, kg 7,000/7,000, freight 7,000/7,000, COD 7,000/7,000, total 7,000/7,000",True
1,Engine reproduces billed_total where billing was right,"4,953/7,000 shipments (70.8%) match to ₹0.01 on every component",True
2,Every mismatch reproduced by a known billing error,"2,047 mismatches; 0 unexplained or inconsistent (billed freight always equals rate card at billed zone/kg)",True
3,Totals identity,"sum(expected) - sum(billed) = 29,695.39 = underbilled 35,986.32 - overbilled 6,290.93",True
4,Billed file is internally consistent,billed_total = billed_freight + billed_cod_fee on all rows,True
5,Cause split adds back to each shipment's gap,"max residual per shipment ₹0.00; causes sum to ₹29,695.39",True
6,Threshold (₹0.005) does not drive the counts,smallest real gap ₹0.94; 0 gaps under ₹0.10; no gaps between ₹0.005 and ₹0.01,True


### Edge cases: how often each occurs in this data, how it's handled, which test covers it

In [11]:
edges = edge_cases(audit)
edges

,edge_case,clause,shipments_in_data,how_handled,tests
0,"Exact-slab weight (e.g. 1.0 kg = 2 slabs, not 3)",A.3,92,"ceil, so exact multiples stay put","test_chargeable_exact_slab_stays, test_freight_exact_slab_weight"
1,Float noise near a slab boundary,A.3,0,"ceil(round(w/0.5, 6)) so 1.0000000002 is not a 3rd slab",test_chargeable_float_noise_does_not_add_a_slab
2,Volumetric heavier than dead weight,A.3,2907,"max(dead, volumetric); volumetric not rounded before max",test_chargeable_uses_higher_of_dead_and_volumetric
3,Declared weight differs from re-measured,§1.2,4164,declared weight never used,test_expected_charge_ignores_declared_weight
4,M11 divisor 4000 changes the slab,M11-§2.2,164,per-merchant divisor,"test_volumetric_m11_divisor_4000, test_m11_divisor_end_to_end"
5,M07 1.0 kg minimum binds,M07-§2.3,398,"max(1.0, rounded weight)","test_chargeable_m07_minimum_one_kg, test_m07_minimum_end_to_end"
6,Zone E (Special Region destination),A.2 rule 2,277,checked before same-state and metro,test_zone_e_beats_b_and_c
7,Same state with a metro origin (Mumbai->Pune/Nagpur),A.2 rule 3 vs 4,64,B before C,test_zone_b_same_state_beats_metro
8,Hyderabad as metro (metro<->Hyderabad),A.2 rule 4,652,Hyderabad in metro list,test_zone_c_metro_to_metro_including_hyderabad
9,"Same-state (Zone B) needs origin state, which is not in the CSV",A.2 rule 3,208,hardcoded city->state map for the 12 pickup cities,test_zone_b_same_state_beats_metro


In [12]:
smallest_gap = audit.loc[audit['diff_total'].abs() > 0.005, 'diff_total'].abs().min()
matched = (audit['reason'] == 'ok').sum()
n_bad = t['underbilled_n'] + t['overbilled_n']
exp_minus_billed = audit['exp_total'].sum() - audit['billed_total'].sum()
e = edges.set_index('edge_case')['shipments_in_data']
q6 = [
    f"Second implementation: a column-wise recomputation from the same contract tables (not calling billing.py) "
    f"agrees with the engine on all {len(audit):,} shipments for zone, kg, freight, COD fee and total.",
    f"Unit tests: {tests['passed']} pass, {tests['failed']} fail; each billing rule is pinned with hand-computed values.",
    f"Reconciliation: the engine reproduces billed_total to ₹0.01 on {matched:,} of {len(audit):,} shipments.",
    f"Every one of the {n_bad:,} mismatches is reproduced exactly by a known billing error (C1-C{len(causes)}, 0 unexplained).",
    f"Totals tie out: sum(expected) - sum(billed) = underbilled - overbilled = {inr_indian(exp_minus_billed)}.",
    f"Threshold-safe: smallest real gap is {inr_indian(smallest_gap)}, far above ₹0.005.",
    f"Edge case - weights: exact-slab weights ({e.iloc[0]}; 1.0 kg = 2 slabs) with a float guard, "
    f"volumetric over dead ({e.iloc[2]:,}), declared weight ignored ({e.iloc[3]:,}), "
    f"M11 divisor 4000 ({e.iloc[4]}), M07 1.0 kg minimum ({e.iloc[5]}).",
    f"Edge case - zones: precedence A > E > B > C > D; Mumbai->Pune/Nagpur = B ({e.iloc[7]}), Hyderabad = metro ({e.iloc[8]}), "
    f"origin state mapped from city (not in CSV).",
    f"Edge case - dates: M02 waiver and M09 discount boundary days ({e.iloc[11]} and {e.iloc[12]} shipments), "
    f"31 Aug waived / 1 Sep charged, 8% to 31 Aug / 12% from 1 Sep.",
    f"Edge case - COD: minimum branch ({e.iloc[13]:,}) vs % branch ({e.iloc[14]:,}), prepaid = 0 ({e.iloc[15]:,}), "
    f"discount never applied to COD fee.",
    f"Multi-error shipments ({e.iloc[-1]}) split across causes without double counting.",
]
print('\n'.join('- ' + s for s in q6))

- Second implementation: a column-wise recomputation from the same contract tables (not calling billing.py) agrees with the engine on all 7,000 shipments for zone, kg, freight, COD fee and total.
- Unit tests: 56 pass, 0 fail; each billing rule is pinned with hand-computed values.
- Reconciliation: the engine reproduces billed_total to ₹0.01 on 4,953 of 7,000 shipments.
- Every one of the 2,047 mismatches is reproduced exactly by a known billing error (C1-C8, 0 unexplained).
- Totals tie out: sum(expected) - sum(billed) = underbilled - overbilled = ₹29,695.39.
- Threshold-safe: smallest real gap is ₹0.94, far above ₹0.005.
- Edge case - weights: exact-slab weights (92; 1.0 kg = 2 slabs) with a float guard, volumetric over dead (2,907), declared weight ignored (4,164), M11 divisor 4000 (164), M07 1.0 kg minimum (398).
- Edge case - zones: precedence A > E > B > C > D; Mumbai->Pune/Nagpur = B (64), Hyderabad = metro (652), origin state mapped from city (not in CSV).
- Edge case - dates: 

## PART 1 ANSWERS (copy-paste)

In [13]:
print('PART 1 ANSWERS')
print('=' * 70)
print(f"1. Total underbilled amount: {inr_indian(t['underbilled_inr'])}  (western {inr_western(t['underbilled_inr'])})")
print(f"2. Number of underbilled shipments: {t['underbilled_n']:,}")
print(f"3. Total overbilled amount: {inr_indian(t['overbilled_inr'])}  (western {inr_western(t['overbilled_inr'])})")
print(f"4. Number of overbilled shipments: {t['overbilled_n']:,}")
print(f"5. Top 3 by net underbilling: {top3(net, 'net_underbilled_inr')}")
print(f"   (by gross underbilling: {top3(gross, 'underbilled_inr')})")
print('\nRoot causes (₹ split across causes; sums to net underbilling):')
for line in root_cause_lines(causes):
    print('  ' + line)
print('\nLooked like leakage but is not:')
for line in not_leakage_lines(leak_free):
    print('  - ' + line)
print('\n6. How I checked my numbers:')
for s in q6:
    print('  - ' + s)
print('=' * 70)
print(f"Sanity: sum(expected) - sum(billed) = {inr_indian(exp_minus_billed)}; "
      f"underbilled - overbilled = {inr_indian(t['underbilled_inr'] - t['overbilled_inr'])}")

PART 1 ANSWERS
1. Total underbilled amount: ₹35,986.32  (western ₹35,986.32)
2. Number of underbilled shipments: 1,493
3. Total overbilled amount: ₹6,290.93  (western ₹6,290.93)
4. Number of overbilled shipments: 554
5. Top 3 by net underbilling: M05 ₹12,671.20; M11 ₹6,207.30; M07 ₹5,878.80
   (by gross underbilling: M05 ₹12,777.42; M07 ₹8,216.10; M11 ₹6,207.30)

Root causes (₹ split across causes; sums to net underbilling):
  C1. Zone: Hyderabad<->metro routes billed Zone D instead of C (Hyderabad is a metro, A.2 rule 4). Affects M01 56, M02 56, M03 48, M05 62, M06 36, M07 394; ship dates 01 Jul - 30 Sep 2026. 652 shipments (240 also have another cause). Impact: overbilled ₹7,505.72 (avg ₹11.51/shipment). e.g. SFX2708195638
  C2. Weight: chargeable weight rounded DOWN to the 0.5 kg slab instead of up (A.3). Affects M05 452; ship dates 01 Jul - 30 Sep 2026. 452 shipments (58 also have another cause). Impact: underbilled ₹10,606.96 (avg ₹23.47/shipment). e.g. SFX2707812567
  C3. Weight:

---
## Supporting detail

### Mismatches by component

In [14]:
(~audit[['zone_ok', 'weight_ok', 'freight_ok', 'cod_ok']]).sum()

zone_ok        652
weight_ok     1281
freight_ok    1832
cod_ok         223
dtype: int64

### Amounts by exact reason combination

In [15]:
by_reason(audit)

,shipments,net_underbilled_inr,merchants
reason,,,
weight rounded down instead of up to 0.5 kg slab (A.3),394,9277.80,M05
volumetric divisor 6000 used instead of contract divisor (A.3),242,6754.52,"M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M11,M12"
divisor 5000 used instead of 4000 (M11-§2.2),153,5422.60,M11
min 0.5 kg used instead of 1.0 kg (M07-§2.3),221,5286.60,M07
"flat INR 35 minimum charged, % of COD value ignored (M08-§3.1)",215,4291.25,M08
"zone D billed, expected C (Hyderabad not treated as metro, A.2 rule 4); min 0.5 kg used instead of 1.0 kg (M07-§2.3)",177,2708.10,M07
volumetric divisor 6000 used instead of contract divisor (A.3) + weight rounded down instead of up to 0.5 kg slab (A.3),17,857.28,M05
"zone D billed, expected C (Hyderabad not treated as metro, A.2 rule 4); weight rounded down instead of up to 0.5 kg slab (A.3)",41,482.22,M05
"volumetric divisor 6000 used instead of contract divisor (A.3); flat INR 35 minimum charged, % of COD value ignored (M08-§3.1)",8,380.30,M08


### Example row per reason

In [16]:
cols = ['awb', 'merchant_id', 'ship_date', 'origin_city', 'dest_city', 'dead_wt_kg', 'declared_wt_kg', 'exp_vol_kg',
        'billed_zone', 'exp_zone', 'billed_chargeable_kg', 'exp_chargeable_kg',
        'billed_freight', 'exp_freight', 'billed_cod_fee', 'exp_cod_fee', 'diff_total', 'reason']
audit[audit['reason'] != 'ok'].groupby('reason', group_keys=False).head(1)[cols]

,awb,merchant_id,ship_date,origin_city,dest_city,dead_wt_kg,declared_wt_kg,exp_vol_kg,billed_zone,exp_zone,billed_chargeable_kg,exp_chargeable_kg,billed_freight,exp_freight,billed_cod_fee,exp_cod_fee,diff_total,reason
1,SFX2700209458,M07,2026-07-14,Hyderabad,Mumbai,0.24,0.17,0.2376,D,C,0.5,1.0,46.80,62.10,49.0,49.00,-15.30,"zone D billed, expected C (Hyderabad not treated as metro, A.2 rule 4); min 0.5 kg used instead of 1.0 kg (M07-§2.3)"
3,SFX2700418916,M03,2026-07-23,Bengaluru,Hyderabad,0.63,0.45,0.6720,D,C,1.0,1.0,72.00,62.10,0.0,0.00,9.90,"zone D billed, expected C (Hyderabad not treated as metro, A.2 rule 4)"
7,SFX2700837832,M05,2026-07-26,Chennai,Mumbai,0.59,0.38,0.5280,C,C,0.5,1.0,42.30,64.86,35.0,35.00,-22.56,weight rounded down instead of up to 0.5 kg slab (A.3)
10,SFX2701152019,M08,2026-07-07,Pune,Warangal,3.75,2.31,2.7840,D,D,4.0,4.0,235.60,235.60,35.0,48.75,-13.75,"flat INR 35 minimum charged, % of COD value ignored (M08-§3.1)"
17,SFX2701885122,M04,2026-07-20,Jaipur,Mumbai,1.75,1.38,4.0920,D,D,3.5,4.5,193.60,242.88,40.0,40.00,-49.28,volumetric divisor 6000 used instead of contract divisor (A.3)
28,SFX2703037141,M03,2026-07-22,Bengaluru,Hyderabad,1.40,1.26,1.5504,D,C,1.5,2.0,97.20,105.30,30.0,30.00,-8.10,"zone D billed, expected C (Hyderabad not treated as metro, A.2 rule 4); volumetric divisor 6000 used instead of cont..."
53,SFX2705655366,M07,2026-07-04,Hyderabad,Indore,0.46,0.46,0.4704,D,D,0.5,1.0,46.80,72.00,0.0,0.00,-25.20,min 0.5 kg used instead of 1.0 kg (M07-§2.3)
83,SFX2708797236,M11,2026-07-24,Lucknow,Delhi,1.08,1.08,1.7955,D,D,1.5,2.0,102.60,129.20,30.0,30.00,-26.60,divisor 5000 used instead of 4000 (M11-§2.2)
166,SFX2708489743,M05,2026-07-18,Chennai,Hyderabad,1.61,1.02,2.4200,D,C,2.0,2.5,127.84,132.54,0.0,0.00,-4.70,"zone D billed, expected C (Hyderabad not treated as metro, A.2 rule 4); weight rounded down instead of up to 0.5 kg ..."
199,SFX2702945800,M05,2026-07-13,Chennai,Ahmedabad,0.66,0.66,2.5200,D,D,2.0,3.0,127.84,180.48,0.0,0.00,-52.64,volumetric divisor 6000 used instead of contract divisor (A.3) + weight rounded down instead of up to 0.5 kg slab (A.3)


### Write the per-shipment audit table to `outputs/audit_by_awb.csv`

In [17]:
write_outputs(audit, ROOT / 'outputs')
sorted(p.name for p in (ROOT / 'outputs').glob('audit_*.csv'))

['audit_by_awb.csv']